In [1]:
print("hello")

hello


In [3]:
%pip install langchain_core langchain-anthropic langgraph langchain-openai


  Using cached langgraph-1.0.3-py3-none-any.whl.metadata (7.8 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-win_amd64.whl.metadata (2.4 kB)
  Using cached tenacity-9.1.2-py3-none-any.whl.metadata (1.2 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached jsonpointer-3.0.0-py2.py3-none-any.whl.metadata (2.3 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached orjson-3.11.4-cp314-cp314-win_amd64.whl.metadata (42 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached zstandard-0.25.0-cp314-cp314-win_amd64.whl.metadata (3.3 kB)
  Using cached anyio-4.11.0-py3-none-any.whl.metadata (4.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached h11-0.16.0-py3-none-any.w


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
%pip install python-dotenv

  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
Using cached python_dotenv-1.2.1-py3-none-any.whl (21 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langsmith import Client, traceable
from pydantic import BaseModel, Field

# -----------------------------------------------------
# Load environment variables
# -----------------------------------------------------
load_dotenv()

os.environ["LANGCHAIN_TRACING"] = "true"   # <-- IMPORTANT (V2 only)
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "MCP-Agent-Demo"

# LangSmith client (optional)
client = Client()

# -----------------------------------------------------
# Initialize LangChain LLM
# -----------------------------------------------------
llm = ChatOpenAI(
    model="gpt-4.1",
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY"),
)

# -----------------------------------------------------
# Structured Output Model
# -----------------------------------------------------
class SearchQuery(BaseModel):
    search_query: str = Field(None)
    justification: str = Field(None)

@traceable  # <---- traced!
def ask_question(q: str):
    structured_llm = llm.with_structured_output(SearchQuery)
    return structured_llm.invoke(q)

# -----------------------------------------------------
# Define Tool
# -----------------------------------------------------
@traceable(run_type="tool")
def multiply(a: int, b: int):
    return a * b

llm_with_tools = llm.bind_tools([multiply])

@traceable(run_type="chain")
def ask_with_tools(q: str):
    return llm_with_tools.invoke(q)

# -----------------------------------------------------
# Run
# -----------------------------------------------------
print("Structured:", ask_question("How does Calcium CT score relate to cholesterol?"))
print("Tool call:", ask_with_tools("What is 2 times 3?").tool_calls)


Structured: search_query='relationship between coronary artery calcium score and cholesterol levels' justification='The user is asking about the connection between calcium CT score (coronary artery calcium score) and cholesterol. To answer accurately, I need to find information on how these two cardiovascular risk factors are related.'
Tool call: [{'name': 'multiply', 'args': {'a': 2, 'b': 3}, 'id': 'call_CV1fNKjvnNSVGjTtZAgT3AiA', 'type': 'tool_call'}]
